In [ ]:
import os
import json

from hera.shared import global_config
from hera.workflows import (
    Artifact,
    DAG,
    Env,
    Parameter,
    Script,
    Volume,
    Workflow,
    script,  # pyright: ignore[reportUnknownVariableType]
)
from hera.workflows import models as m
from hera.workflows.archive import NoneArchiveStrategy

# Sets the default image, unless specified otherwise, to this.
global_config.set_class_defaults(  # pyright: ignore
    Script, image="ghcr.io/diamondlightsource/httomo:latest"
)
# Do note! This workflow will not run, as the DLS paths have been replaced with
# placeholders, so as not to fill up required directories.

In [ ]:
# The script decorator allows hera to convert python code into yaml
@script(
# Assure that we only run on a pod with certain parameters
        pod_spec_patch=json.dumps({"containers":
                    [{"name":"main",
                    "resources":
                        {"limits":{"cpu":"{{inputs.parameters.nprocs}}",
                                    "memory":"{{inputs.parameters.memory}}",
                                    "nvidia.com/gpu":"{{inputs.parameters.nprocs}}"},
                        "requests":{"cpu":"{{inputs.parameters.nprocs}}",
                                    "memory":"{{inputs.parameters.memory}}",
                                    "nvidia.com/gpu":"{{inputs.parameters.nprocs}}"}}}]}),

# Assure that we only run on a pod with certain parameters
            tolerations=[
                m.Toleration(key="nvidia.com/gpu",operator="Exists",effect="NoSchedule"),
                m.Toleration(key="nodetype",operator="Equal",value="gpu",effect="NoSchedule"),
                m.Toleration(key="nodegroup",operator="Equal",value="workflows",effect="NoSchedule")],
# Inform the pod which installation of python we intend to run on
    command=["/opt/conda/bin/python"],
# mount both volumes - one being a parameters in the diamond file system, the other
# being a mounted file path
    volume_mounts=[
        m.VolumeMount(name="session", mount_path="{{workflow.parameters.visitdir}}"),
        m.VolumeMount(name="tmpdir", mount_path="/tmp"),
    ],
# inform the next step that the file we produce here will be used as a parameter for the
# next script
    outputs=[
        Parameter(
            name="out-path",
            value_from=m.ValueFrom(
                path="/tmp/parameters.json"
            )
        )
    ],
# establish some environmental variables shared between steps.
    env=[
        Env(name="CUPY_CACHE_DIR", value="/tmp/.cupy/kernel_cache"),
        Env(name="MKL_NUM_THREADS", value="1"),
        Env(name="NUMEXPR_NUM_THREADS", value="1"),
        Env(name="OMP_NUM_THREADS", value="1"),
    ]
)
def tomo_recon(
    config: str, input: str, output: str, recon_outdir_name: str, nprocs: int, memory: str
):
    import json
    import subprocess
# load the config string as a json - otherwise, in the yaml conversion, this is loaded
# as a string instead, which prevents mpirun from functioning.
    loaded_config = json.dumps(config)

    subprocess.check_call([
        "/opt/conda/bin/mpirun",
        "-n",
        str(nprocs),
        "/opt/conda/bin/python",
        "-m",
        "httomo",
        "run",
        "--pipeline-format",
        "json",
        "--output-folder-name",
        recon_outdir_name,
        input,
        loaded_config,
        output
    ])

    with open("/tmp/parameters.json", "w") as f:
        json.dump(f"{output}/{recon_outdir_name}", f)

In [4]:
@script(
    command=["/opt/conda/bin/python"],
    volume_mounts=[
        m.VolumeMount(name="session", mount_path="{{workflow.parameters.visitdir}}"),
        m.VolumeMount(name="tmpdir", mount_path="/tmp"),
    ],
    #our outputs this time are artifacts, that we do not want compressed.
    outputs=[
        Artifact(
            name="recon",
            path="{{inputs.parameters.tmpdir_path}}/{{inputs.parameters.raw_recon_filename}}",
            archive=NoneArchiveStrategy(),
        ),
        Artifact(
            name="metadata",
            path="{{inputs.parameters.tmpdir_path}}/{{inputs.parameters.metadata_filename}}",
            archive=NoneArchiveStrategy(),
        ),
    ]
)
def convert_recon_data_format(recon_dir_path: str,
                              tmpdir_path: str = "/tmp",
                              raw_recon_filename: str = "recon.raw",
                              metadata_filename: str = "metadata.json"):
    import json
    from pathlib import Path

    import h5py



    RAW_RECON_PATH = f"{tmpdir_path}/{raw_recon_filename}"
    HDF5_RECON_DIR = Path(recon_dir_path)
    HDF5_RECON_FILENAME_PATTERN = "*-httomolib-rescale_to_int.h5"
    hdf5_recon_data_path = list(HDF5_RECON_DIR.glob(HDF5_RECON_FILENAME_PATTERN))[0]

    with h5py.File(hdf5_recon_data_path, "r") as f:
        data = f["/data"][:]
        data.tofile(RAW_RECON_PATH)

    METADATA_PATH = f"{tmpdir_path}/{metadata_filename}"

    order = "C" if data.flags.c_contiguous else "F"
    metadata = {"shape": list(data.shape), "dtype": str(data.dtype), "order": order}
    with open(METADATA_PATH, "w") as f:
        f.write(json.dumps(metadata, indent=2))

In [ ]:
from hera.workflows.volume import HostPathVolume

with Workflow(
    # name of workflow - will append a short identifier automatically.
    name="visr-recon-with-python-interface-to-workflows",
    # the following is the same as for writing any yaml workflow, but as variables.
    # All of these are required aside from "workflows.argoproj.io/description".
    entrypoint="workflowentry",
    api_version="argoproj.io/v1alpha1",
    kind="WorkflowTemplate",
    labels={"workflows.diamond.ac.uk/science-group-imaging": "true"},
    annotations={
        "workflows.argoproj.io/title": "ViSR recon",
        "workflows.argoproj.io/description": """ViSR recon
example.yaml""",
        "workflows.diamond.ac.uk/repository": "https://github.com/DiamondLightSource/python-interface-to-workflows",
    },
    # mounts the diamond file system
    volumes=[
        Volume(name="tmpdir", mount_path="/tmp/", size="1Gi"),
        HostPathVolume(name="session",
                       path="{{workflow.parameters.visitdir}}",
                       type="Directory"),
    ],
    arguments=m.Arguments(
        parameters=[
            m.Parameter(
                name="visitdir",
                value_from=m.ValueFrom(
                    config_map_key_ref=m.ConfigMapKeySelector(
                        name="sessionspaces",
                        key="data_directory"
                    )
                )
            )
        ]
    )
) as w:
    with DAG(name="workflowentry"):
        # Alternatively, we could have this in /mounted_files/ and run
        # a simple step where we copy over this json, before running
        # the step in the httomo image
        config = """[
  {
    "method": "standard_tomo",
    "module_path": "httomo.data.hdf.loaders",
    "parameters": {
      "data_path": "/entry1/tomo_entry/data/data",
      "image_key_path": "/entry1/tomo_entry/instrument/detector/image_key",
      "rotation_angles": {
        "data_path": "/entry1/tomo_entry/data/rotation_angle"
      },
      "preview": {
        "detector_y": {
          "start": 100,
          "stop": 102
        }
      }
    }
  },
  {
    "method": "remove_outlier",
    "module_path": "tomopy.misc.corr",
    "parameters": {
      "dif": 0.1,
      "size": 3,
      "axis": "auto"
    }
  },
  {
    "method": "dark_flat_field_correction",
    "module_path": "httomolibgpu.prep.normalize",
    "parameters": {
        "flats_multiplier": 1,
        "darks_multiplier": 1
    }
  },
  {
    "method": "find_center_vo",
    "module_path": "httomolibgpu.recon.rotation",
    "parameters": {
      "ind": null,
      "smin": -50,
      "smax": 50,
      "srad": 6,
      "step": 0.25,
      "ratio": 0.5,
      "drop": 20
    },
    "id": "centering",
    "side_outputs": {
      "cor": "centre_of_rotation"
    }
  },
  {
    "method": "FBP3d_tomobar",
    "module_path": "httomolibgpu.recon.algorithm",
    "parameters": {
      "center": "${{centering.side_outputs.centre_of_rotation}}",
      "filter_freq_cutoff": 0.6,
      "recon_size": null,
      "recon_mask_radius": null
    },
    "save_result": true
  },
  {
      "method": "calculate_stats",
      "module_path": "httomo.methods",
      "parameters": {},
      "id": "statistics",
      "side_outputs": {
          "glob_stats": "glob_stats"
      }
  },
  {
      "method": "rescale_to_int",
      "module_path": "httomolib.misc.rescale",
      "parameters": {
          "perc_range_min": 0,
          "perc_range_max": 100,
          "bits": 8,
          "glob_stats": "${{statistics.side_outputs.glob_stats}}"
      },
      "save_result": true
  }
]"""
        recon = tomo_recon(
            arguments={
                "config": config,
                # This is the input file, a .nxs
                "input": "/dls/i12/data/2025/.......",
                # This can be any folder on /dls/ where we wish to put the file.
                "output": "/dls/i12/data/2025/........",
                "recon_outdir_name": "sweep-run", "nprocs": 1, "memory": "1Gi"
            }
        )
        convert = convert_recon_data_format(
            arguments={
                "recon_dir_path": recon.get_parameter("out-path"),
            }
        )
        #run step recon before running step convert
        recon >> convert  # pyright: ignore




In [10]:
#Firstly we write the yaml file we intend to send to GraphQL...
with open("/workspaces/python-interface-to-workflows/src/python_interface_to_workflows/templates/visr_recon.txt", "w") as div:
    div.write(w.to_yaml())  # pyright: ignore[reportUnknownMemberType]

In [ ]:
# Then we run this to submit the workflow to GraphQL, first linting the yaml.
from python_workflow_submitter.submit_workflow import submit_workflow_yaml

await submit_workflow_yaml("/workspaces/imaging-python-workflows/src/imaging_python_workflows/templates/visr_recon.yaml")